# Agent Memory: Adding Persistent Conversational Memory

This notebook demonstrates how to add memory to agents, enabling them to remember past interactions and maintain context across multiple queries.

## Learning Objectives
- Understand different types of agent memory
- Implement in-memory and persistent memory
- Use SQLite for conversation checkpointing
- Manage conversation threads and sessions

## Related Theoretical Content
- [Multi-Agent AI](../../notes/04-multi_agent_ai/README.md)
- [Agent Memory Systems](../../notes/04-multi_agent_ai/02-agent_memory/README.md)

## Prerequisites
This notebook is self-contained and can run independently.

## Setup: Import Dependencies

Load required libraries and tools.

In [ ]:
import os
import random
import sqlite3
import yfinance as yf
from api_keys import get_api_key

print(" Dependencies imported")

## 1. Create Tools (Same as Before)

Define the same tools from the previous notebook.

In [ ]:
from langchain.tools import tool
from langchain_tavily import TavilySearch

os.environ["TAVILY_API_KEY"] = get_api_key('tavily')

@tool
def web_search(query: str) -> str:
    """Use this tool to fetch latest info from the web."""
    print(f'� Search: {query}')
    search = TavilySearch(max_results=3)
    return search.invoke(query)

@tool
def random_number_generator(start: int = 1, end: int = 100) -> int:
    """Generate a random number between start and end."""
    print('� Random number generated')
    return random.randint(start, end)

@tool
def get_current_user() -> str:
    """Get the current user's name."""
    return os.getlogin()

@tool
def get_stock_price(ticker: str) -> str:
    """Get current stock price. Input must be a ticker symbol (e.g., MSFT)."""
    try:
        print(f"� Stock price: {ticker}")
        data = yf.Ticker(ticker)
        price = data.history(period="1d")['Close'].iloc[-1]
        return f"The current price for {ticker} is ${price:.2f}"
    except Exception as e:
        return f"Could not find price for {ticker}"

tools = [web_search, random_number_generator, get_current_user, get_stock_price]
print(f" {len(tools)} tools created")

## 2. Initialize LLM

Set up the language model.

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=get_api_key('groq')
)

print(" LLM initialized")

## 3. Create Agent with In-Memory Checkpointer

**Memory Types:**

1. **InMemorySaver**: Stores conversation in RAM (lost when program ends)
2. **SqliteSaver**: Persists to SQLite database (survives restarts)

We'll start with in-memory and then upgrade to persistent storage.

In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.checkpoint.memory import InMemorySaver

# Create memory checkpointer
memory = InMemorySaver()

# Define system prompt
system_prompt = '''
You are a helpful information assistant with memory.

IMPORTANT:
- You can remember previous interactions in this conversation
- Reference past queries and answers when relevant
- Greet returning users by name if you've met before
- Keep responses concise (under 40 lines)

Tools available:
- web_search: Latest web information
- random_number_generator: Generate random numbers
- get_current_user: Get user's name
- get_stock_price: Get stock prices by ticker
'''

# Create prompt with chat history
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Create agent
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    memory=memory
)

print(" Agent with in-memory checkpointing created")

## 4. Test Memory: Multi-Turn Conversation

The agent should remember information from previous messages in the same thread.

In [ ]:
# Conversation Thread 1
thread_id = "user_session_1"
config = {"configurable": {"thread_id": thread_id}}

print("Starting conversation in thread: user_session_1\n")

# Turn 1: Ask for stock price
result1 = agent_executor.invoke(
    {"input": "What's the stock price of Microsoft?"},
    config=config
)
print(f"\nAgent: {result1['output']}\n")
print("="*60)

In [ ]:
# Turn 2: Follow-up question (should remember the previous stock)
result2 = agent_executor.invoke(
    {"input": "What about Apple's price?"},
    config=config
)
print(f"\nAgent: {result2['output']}\n")
print("="*60)

In [ ]:
# Turn 3: Reference to previous information
result3 = agent_executor.invoke(
    {"input": "Which one was more expensive?"},
    config=config
)
print(f"\nAgent: {result3['output']}\n")
print("="*60)

## 5. Multiple Conversation Threads

Different thread IDs create separate conversation contexts. This is useful for:
- Multiple users
- Different topics
- Isolated contexts

In [ ]:
# Start a NEW thread (separate context)
thread_id_2 = "user_session_2"
config_2 = {"configurable": {"thread_id": thread_id_2}}

print("Starting NEW conversation in thread: user_session_2\n")

# This agent should NOT remember the Microsoft/Apple discussion
result_new = agent_executor.invoke(
    {"input": "What stocks have we discussed?"},
    config=config_2
)
print(f"\nAgent: {result_new['output']}\n")
print("(Notice: No memory of previous thread)")

## 6. Persistent Memory with SQLite

**SqliteSaver** writes conversations to disk. Benefits:
- Survives program restarts
- Can be backed up
- Enables conversation analytics
- Supports long-term user memory

In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver

# Create or connect to SQLite database
db_path = ".data/agent_checkpoints.db"
os.makedirs(".data", exist_ok=True)

conn = sqlite3.connect(db_path, check_same_thread=False)
persistent_memory = SqliteSaver(conn)

# Create agent with persistent memory
agent_persistent = create_tool_calling_agent(llm, tools, prompt)
agent_executor_persistent = AgentExecutor(
    agent=agent_persistent,
    tools=tools,
    verbose=True,
    memory=persistent_memory
)

print(f" Agent with persistent SQLite memory created")
print(f"  Database: {db_path}")

## 7. Test Persistent Memory

Conversations saved to SQLite will persist even if you restart the notebook.

In [ ]:
# Create a persistent conversation
persistent_thread = "persistent_session_1"
persistent_config = {"configurable": {"thread_id": persistent_thread}}

print("Starting persistent conversation\n")

result_p1 = agent_executor_persistent.invoke(
    {"input": "My favorite stock is Tesla. Remember this."},
    config=persistent_config
)
print(f"\nAgent: {result_p1['output']}\n")

In [ ]:
# Continue conversation (will be saved to DB)
result_p2 = agent_executor_persistent.invoke(
    {"input": "What's my favorite stock?"},
    config=persistent_config
)
print(f"\nAgent: {result_p2['output']}\n")
print("(This preference is now saved to SQLite)")

## 8. Inspect Stored Conversations

You can query the SQLite database to see stored conversations.

In [ ]:
# Query the database
cursor = conn.cursor()

# Check what tables exist
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("Tables in checkpoint database:")
for table in tables:
    print(f"  - {table[0]}")

    # Count records in each table
    cursor.execute(f"SELECT COUNT(*) FROM {table[0]}")
    count = cursor.fetchone()[0]
    print(f"    Records: {count}")

## 9. Clear Memory (Optional)

Sometimes you need to clear conversation history.

In [ ]:
# To clear a specific thread, you would delete from the database
# For demonstration, we'll show how to close the connection

def clear_thread(thread_id: str):
    """Clear a specific conversation thread."""
    # Implementation depends on checkpoint structure
    print(f"Would clear thread: {thread_id}")
    # cursor.execute("DELETE FROM checkpoints WHERE thread_id = ?", (thread_id,))
    # conn.commit()

# Example (not executed)
# clear_thread("persistent_session_1")

## Key Takeaways

1. **Memory Types**:
   - **InMemorySaver**: Fast, temporary, lost on restart
   - **SqliteSaver**: Persistent, survives restarts, queryable

2. **Thread Management**: Use thread IDs to separate conversations

3. **Context Retention**: Agents can reference previous exchanges

4. **Use Cases**:
   - Customer support (remember user issues)
   - Personal assistants (learn user preferences)
   - Multi-turn reasoning (complex problem solving)

5. **Limitations**:
   - Memory grows over time (need pruning strategies)
   - Long conversations may exceed context limits
   - No semantic compression (stores raw messages)

## Best Practices

- Use **InMemorySaver** for: Testing, temporary sessions, low latency
- Use **SqliteSaver** for: Production, user history, analytics
- Implement **memory pruning** for long conversations
- Consider **message summarization** for efficiency

## Next Steps

- [07-langgraph.ipynb](07-langgraph.ipynb): Build complex agent workflows with state machines